# Capítulo 1 · Cómo aprende un modelo

Una recta que aprende a estimar precios ficticios de viviendas corrigiendo sus dos parámetros con descenso de gradiente. Python estándar y CPU: no hay que instalar nada. Ejecuta las celdas en orden.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio: el mismo programa, por partes, para ejecutarlo y experimentar.

## 1. Los datos y la predicción

Tres viviendas de entrenamiento y una reservada. `entrada` cuenta la superficie en bloques de 50 m² y `predecir` aplica la regla ŷ = wx + b, en miles de euros.

In [1]:
"""Capítulo 1: precios ficticios de viviendas. Python estándar y CPU."""
import math

# Superficie en m²; precio en miles de euros.
VIVIENDAS = [(50.0, 150.0), (100.0, 250.0), (150.0, 350.0)]
RESERVADA = (75.0, 200.0)


def entrada(superficie):
    return superficie / 50.0


def predecir(superficie, w, b):
    return w * entrada(superficie) + b

In [2]:
print("predecir(100, 50, 50) =", predecir(100, 50, 50))
for superficie, precio in VIVIENDAS:
    estimacion = predecir(superficie, 50, 50)
    print(f"{superficie:g} m²: precio {precio:g}, estimación {estimacion:g}, error {estimacion - precio:g}")

predecir(100, 50, 50) = 150.0
50 m²: precio 150, estimación 100, error -50
100 m²: precio 250, estimación 150, error -100
150 m²: precio 350, estimación 200, error -150


## 2. La pérdida y los gradientes

`perdida` es el error cuadrático medio. `gradientes` suma la aportación de cada vivienda: `2 * error * x` para el peso y `2 * error` para el sesgo, y divide entre el número de viviendas.

In [3]:
def perdida(viviendas, w, b):
    errores = [predecir(s, w, b) - precio for s, precio in viviendas]
    return sum(e ** 2 for e in errores) / len(errores)


def gradientes(viviendas, w, b):
    dw, db = 0.0, 0.0
    for superficie, precio in viviendas:
        x = entrada(superficie)
        error = w * x + b - precio
        dw += 2 * error * x
        db += 2 * error
    n = len(viviendas)
    return dw / n, db / n

Comprueba las cifras del capítulo: la pérdida inicial, la de w = 60 y las pruebas con w = 49 y w = 51. La diferencia entre estas dos, dividida entre 2, coincide con el gradiente del peso.

In [4]:
for w in (50, 60, 49, 51):
    print(f"w = {w}, b = 50: pérdida {perdida(VIVIENDAS, w, 50):.2f}")
dw, db = gradientes(VIVIENDAS, 50, 50)
print(f"Gradientes en el punto inicial: peso {dw:.2f}, sesgo {db:.2f}")
print(f"(L(51) - L(49)) / 2 = {(perdida(VIVIENDAS, 51, 50) - perdida(VIVIENDAS, 49, 50)) / 2:.2f}")

w = 50, b = 50: pérdida 11666.67
w = 60, b = 50: pérdida 7466.67
w = 49, b = 50: pérdida 12138.00
w = 51, b = 50: pérdida 11204.67
Gradientes en el punto inicial: peso -466.67, sesgo -200.00
(L(51) - L(49)) / 2 = -466.67


## 3. Un paso a mano

Una actualización con tasa 0,05: los dos gradientes se calculan con los parámetros antiguos y después se actualizan los dos.

In [5]:
tasa = 0.05
w, b = 50.0, 50.0
dw, db = gradientes(VIVIENDAS, w, b)
w, b = w - tasa * dw, b - tasa * db
print(f"Tras un paso: w = {w:.2f}, b = {b:.2f}, pérdida {perdida(VIVIENDAS, w, b):.2f}")

Tras un paso: w = 73.33, b = 60.00, pérdida 2351.85


## 4. Entrenar

`entrenar` repite el paso y guarda en el historial el número de actualizaciones, el peso, el sesgo y la pérdida.

In [6]:
def entrenar(viviendas, tasa=0.05, pasos=1000):
    w, b = 50.0, 50.0
    historial = [(0, w, b, perdida(viviendas, w, b))]
    for paso in range(1, pasos + 1):
        dw, db = gradientes(viviendas, w, b)
        w, b = w - tasa * dw, b - tasa * db
        if not all(math.isfinite(v) and abs(v) < 1e100 for v in (w, b)):
            raise ValueError("El ajuste diverge: reduce la tasa.")
        historial.append((paso, w, b, perdida(viviendas, w, b)))
    return w, b, historial

## 5. El experimento completo

`comparar_tasas` entrena desde el mismo punto con tres tasas y veinte pasos. `ejecutar` entrena 1000 pasos, estima la vivienda reservada y compara las tasas.

In [7]:
def comparar_tasas(tasas=(0.005, 0.05, 0.3), pasos=20):
    for tasa in tasas:
        _, _, historial = entrenar(VIVIENDAS, tasa=tasa, pasos=pasos)
        print(f"Tasa {tasa}: pérdida tras {pasos} pasos {historial[-1][3]:.6g}")


def ejecutar():
    w, b, historial = entrenar(VIVIENDAS)
    print("Paso      Peso     Sesgo  Pérdida")
    for paso, wp, bp, valor in historial:
        if paso in (0, 1, 2, 20, 100, 1000):
            print(f"{paso:4d} {wp:9.4f} {bp:9.4f}  {valor:.6g}")
    superficie, precio = RESERVADA
    print(f"Vivienda reservada de {superficie:g} m²: estimación "
          f"{predecir(superficie, w, b):.2f}, precio {precio:g} (miles de euros)")
    comparar_tasas()
    return historial

In [8]:
historial = ejecutar()

Paso      Peso     Sesgo  Pérdida
   0   50.0000   50.0000  11666.7
   1   73.3333   60.0000  2351.85
   2   83.7778   64.3333  503.453
  20   93.6347   64.4699  30.0364
 100   97.5807   55.4996  4.33887
1000  100.0000   50.0001  1.52871e-09
Vivienda reservada de 75 m²: estimación 200.00, precio 200 (miles de euros)
Tasa 0.005: pérdida tras 20 pasos 1231.81
Tasa 0.05: pérdida tras 20 pasos 30.0364
Tasa 0.3: pérdida tras 20 pasos 5.54075e+18


## 6. Experimenta con la tasa

Prueba otras tasas con veinte pasos (ejercicio 9): 0.1, 0.15, 0.2… Escribe antes qué esperas que ocurra. Si una tasa hace que los parámetros se disparen, el programa se detiene con un aviso.

In [9]:
comparar_tasas(tasas=(0.1, 0.15, 0.2))

Tasa 0.1: pérdida tras 20 pasos 18.4081
Tasa 0.15: pérdida tras 20 pasos 11.2142
Tasa 0.2: pérdida tras 20 pasos 3.15795e+07


## Ejercicios

1. Explica con tus palabras qué significan el peso y el sesgo en el ejemplo. ¿Qué diferencia hay entre cambiar la superficie de una vivienda y cambiar el peso del modelo?

2. Una vivienda tiene 125 m². Convierte su superficie a la entrada x y calcula su precio estimado con w=50, b=50. Expresa el resultado en euros.

3. Reconstruye los tres errores iniciales y su pérdida media. Explica por qué la pérdida no está expresada directamente en euros.

4. Aumenta solo el peso de 50 a 60 y conserva el sesgo. Calcula las tres predicciones y comprueba si la pérdida mejora.

5. Calcula las aportaciones 2eᵢxᵢ de cada vivienda al gradiente del peso en el punto inicial. Haz lo mismo con las aportaciones 2eᵢ del sesgo y obtén los promedios.

6. Aplica una actualización con tasa 0,05. Interpreta el peso y el sesgo nuevos y calcula la pérdida resultante.

7. ¿Por qué el sesgo puede aumentar en el primer paso aunque los datos se hayan creado con sesgo 50?

8. Ejecuta el programa y comprueba las tres tasas de la tabla del capítulo. Explica la diferencia entre avanzar despacio y divergir.

9. Prueba las tasas 0,1, 0,15 y 0,2 con veinte pasos, en el cuaderno o cambiando `comparar_tasas`. ¿Cuál baja más? ¿A partir de qué tasa diverge?

10. Estima el precio de la vivienda reservada de 75 m² usando los parámetros finales. ¿Es interpolación o extrapolación? ¿Qué límites tiene esta comprobación?

11. Explica la diferencia entre entrenamiento e inferencia utilizando el código del capítulo.

12. Dos viviendas tienen 100 m², pero una cuesta 200 000 euros y otra 300 000. ¿Puede este modelo acertar exactamente ambas? Razona qué información podría faltar.

13. Describe en 100–150 palabras qué comparte este ejemplo con entrenar un modelo de lenguaje y menciona al menos tres diferencias.

## Referencias

Inspiración: [primera lección](https://www.youtube.com/watch?v=VMj-3S1tku0) de [Zero to Hero](https://github.com/karpathy/nn-zero-to-hero), de Andrej Karpathy, con su motor de derivadas [micrograd](https://github.com/karpathy/micrograd). Datos de viviendas, código y explicaciones propios.